# How To Eliminate Loops From Your Python Code

Companion to the [complete article](https://todatabeyond.com/blog/how-to-eliminate-loops-from-your-python-code).

Run all cells in order on Python 3.10+. CPU only, no API keys. Setup installs tested package versions; a later example downloads the original author repository’s Pokémon CSV at a pinned commit. No local dataset prerequisite, saved outputs or execution counts.

All sixteen original gist examples are retained. The nested-pair example corrects source indentation, and the row-mean section explicitly converts the statistics into a NumPy array. Bounded standard-library timeit calls replace IPython timing magics so the examples also work as Python code. Timing functions return results rather than printing thousands of times; the two original visible attack-message examples still print their complete messages.

Timings vary with hardware, interpreter and input size; no ordering is asserted. A comprehension or map still iterates, and NumPy is most useful when compiled array work amortizes call overhead. The source describes a 720-Pokémon dataset, but the actual linked CSV includes additional forms; the notebook reports its real row count. Original article prose and timing screenshots are preserved.


In [ ]:
%pip install --quiet numpy==2.3.3 pandas==2.3.3


## 1. Setup and original three-row statistics


In [ ]:
import timeit
import numpy as np

def benchmark(function, number=1000, repeat=5):
    samples = timeit.repeat(function, number=number, repeat=repeat)
    print(f"Best of {repeat}: {min(samples) / number * 1e6:.3f} microseconds per call")

# List of HP, Attack, Defense, Speed
poke_stats = [
[90, 92, 75, 60],
[25, 20, 15, 90],
[65, 130, 60, 75],
]
print(poke_stats)


## 2. Row totals with a loop


In [ ]:
def row_totals_loop():
    totals = []
    for row in poke_stats:
        totals.append(sum(row))
    return totals

print(row_totals_loop())
benchmark(row_totals_loop)


## 3. Row totals with a comprehension


In [ ]:
def row_totals_comprehension():
    totals_comp = [sum(row) for row in poke_stats]
    return totals_comp

print(row_totals_comprehension())
benchmark(row_totals_comprehension)


## 4. Row totals with map


In [ ]:
def row_totals_map():
    totals_map = [*map(sum, poke_stats)]
    return totals_map

print(row_totals_map())
benchmark(row_totals_map)


## 5. Unique pairs with nested loops


In [ ]:
poke_types = ['Bug', 'Fire', 'Ghost', 'Grass', 'Water']

# Nested for loop approach
combos = []
for x in poke_types:
    for y in poke_types:
        if x == y:
            continue
        if ((x,y) not in combos) & ((y,x) not in combos):
            combos.append((x,y))

print(combos)


## 6. Unique pairs with itertools


In [ ]:
# Built-in module approach
from itertools import combinations
combos2 = [*combinations(poke_types, 2)]

print(combos2)


## 7. Row means with a loop


In [ ]:
poke_stats = np.asarray(poke_stats)

def row_averages_loop():
    avgs = []
    for row in poke_stats:
        avg = np.mean(row)
        avgs.append(avg)
    return avgs

print([float(value) for value in row_averages_loop()])
benchmark(row_averages_loop)


## 8. Repeat the row-mean loop benchmark


In [ ]:
def row_averages_loop_timed():
    avgs = []
    for row in poke_stats:
        avg = np.mean(row)
        avgs.append(avg)
    return avgs

print([float(value) for value in row_averages_loop_timed()])
benchmark(row_averages_loop_timed)


## 9. Axis-wise NumPy mean


In [ ]:
def row_averages_numpy():
    avgs = poke_stats.mean(axis=1)
    return avgs

print(row_averages_numpy().tolist())
benchmark(row_averages_numpy)


## 10. Mean inside the attack loop


In [ ]:
import numpy as np

names = ['Absol', 'Aron', 'Jynx', 'Natu', 'Onix']
attacks = np.array([130, 70, 50, 50, 45])

for pokemon,attack in zip(names, attacks):
    total_attack_avg = attacks.mean()
    if attack > total_attack_avg:
        print(
            "{}'s attack: {} > average: {}!"
            .format(pokemon, attack, total_attack_avg)
        )


## 11. Mean outside the attack loop


In [ ]:
import numpy as np

names = ['Absol', 'Aron', 'Jynx', 'Natu', 'Onix']
attacks = np.array([130, 70, 50, 50, 45])

# Calculate total average once (outside the loop)
total_attack_avg = attacks.mean()

for pokemon,attack in zip(names, attacks):
    if attack > total_attack_avg:
        print(
            "{}'s attack: {} > average: {}!"
            .format(pokemon, attack, total_attack_avg)
        )


## 12. Benchmark repeated attack mean


In [ ]:
def attack_messages_inside():
    messages = []
    for pokemon,attack in zip(names, attacks):
        total_attack_avg = attacks.mean()
        if attack > total_attack_avg:
            messages.append(
                "{}'s attack: {} > average: {}!"
                .format(pokemon, attack, total_attack_avg)
            )
    return messages

print("\n".join(attack_messages_inside()))
benchmark(attack_messages_inside)


## 13. Benchmark hoisted attack mean


In [ ]:
def attack_messages_outside():
    messages = []
    # Calculate total average once (outside the loop)
    total_attack_avg = attacks.mean()
    for pokemon,attack in zip(names, attacks):
        if attack > total_attack_avg:
            messages.append(
                "{}'s attack: {} > average: {}!"
                .format(pokemon, attack, total_attack_avg)
            )
    return messages

print("\n".join(attack_messages_outside()))
benchmark(attack_messages_outside)


## 14. Load the original Pokémon data


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import pandas as pd

POKEMON_URL = "https://raw.githubusercontent.com/youssefHosni/Advanced-Python-Programming-Tutorials-/2507c7f72e3cb08ef349047a4ca052bcc7307337/Datasets/pokemon.csv"
csv_path = Path("pokemon-loop-examples.csv")
urlretrieve(POKEMON_URL, csv_path)
pokemon = pd.read_csv(csv_path)
names_list = pokemon['Name']
legend_status_list = pokemon['Legendary']
generations_list = pokemon['Generation']
print("Dataset shape:", pokemon.shape)
print("SHA-256:", hashlib.sha256(csv_path.read_bytes()).hexdigest())
print(pokemon[['Name', 'Legendary', 'Generation']].head(3).to_string(index=False))


## 15. Convert each tuple in the loop


In [ ]:
def convert_inside():
    poke_data = []
    for poke_tuple in zip(names_list, legend_status_list, generations_list):
        poke_list = list(poke_tuple)
        poke_data.append(poke_list)
    return poke_data

print(convert_inside()[:3])
print("Rows:", len(convert_inside()))
benchmark(convert_inside, number=100)


## 16. Convert the collected tuples with map


In [ ]:
def convert_after():
    poke_data_tuples = []
    for poke_tuple in zip(names_list, legend_status_list, generations_list):
        poke_data_tuples.append(poke_tuple)
        
    poke_data = [*map(list, poke_data_tuples)]
    return poke_data

print(convert_after()[:3])
print("Rows:", len(convert_after()))
benchmark(convert_after, number=100)


## Regression checks

Check exact output equivalence, all unordered pairs, empty inputs and value variations. Runtime is measured, never asserted.


In [ ]:
# Every totals and row-mean implementation returns the same values.
assert row_totals_loop() == row_totals_comprehension() == row_totals_map() == [317, 150, 330]
np.testing.assert_allclose(row_averages_loop(), [79.25, 37.5, 82.5])
np.testing.assert_allclose(row_averages_loop_timed(), row_averages_numpy())

# The corrected nested loop and itertools produce all ten unordered pairs.
assert len(combos) == len(combos2) == 10
assert {frozenset(pair) for pair in combos} == {frozenset(pair) for pair in combos2}
assert all(len(set(pair)) == 2 for pair in combos)

# Moving the invariant mean changes no selected names or messages.
expected_messages = ["Absol's attack: 130 > average: 69.0!", "Aron's attack: 70 > average: 69.0!"]
assert attack_messages_inside() == attack_messages_outside() == expected_messages

# Both tuple/list approaches retain every original record and Python list type.
inside = convert_inside()
after = convert_after()
assert inside == after
assert len(inside) == len(pokemon)
assert all(isinstance(row, list) and len(row) == 3 for row in inside)
assert inside[0] == [names_list.iloc[0], legend_status_list.iloc[0], generations_list.iloc[0]]

# Check empty inputs and broader row values without relying on timing rankings.
saved_stats = poke_stats
try:
    poke_stats = np.empty((0, 4))
    assert row_totals_loop() == row_totals_comprehension() == row_totals_map() == []
    assert row_averages_loop() == []
    assert row_averages_numpy().shape == (0,)
    poke_stats = np.array([[0, 0, 0, 0], [-2, 4, 6, 8], [7, 7, 7, 7]])
    np.testing.assert_allclose(row_totals_loop(), poke_stats.sum(axis=1))
    np.testing.assert_allclose(row_totals_comprehension(), row_totals_map())
    np.testing.assert_allclose(row_averages_loop(), row_averages_numpy())
finally:
    poke_stats = saved_stats

assert list(map(list, zip([], [], []))) == []
assert list(map(list, zip(['one', 'two'], [False], [1, 2]))) == [['one', False, 1]]
print("All totals, combinations, means, attack and tuple-conversion regression checks passed.")
print("Original dataset rows:", len(pokemon))


## Original and official references

- [Original author code and dataset](https://github.com/youssefHosni/Advanced-Python-Programming-Tutorials-)
- [Python timeit](https://docs.python.org/3/library/timeit.html)
- [itertools.combinations](https://docs.python.org/3/library/itertools.html#itertools.combinations)
- [NumPy mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)
